# 02.2 — Alpine ski resort scraping and data enrichment

This notebook builds a balanced sample of **70 ski resorts in each of Switzerland, Italy, Austria and France (280 total)**. The source is Skiresort.com's dedicated Alpine directory and its country-specific Alpine filters—not the general national ski-resort pages. This keeps the sample within the Alps and excludes areas in the Pyrenees, Apennines and other mountain ranges.

The workflow has five stages: discover Alpine listings, parse the listing cards, enrich each selected resort from its detail and snow-reliability pages, select a transparent balanced sample, and validate/export the result. Missing source values remain missing; no resort or price is fabricated.

In [ ]:
import hashlib
import json
import os
import re
import time
import unicodedata
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.skiresort.com"
RAW_DIR = "../data/raw/skiresort_alpine_pages"
DETAIL_DIR = "../data/raw/skiresort_alpine_details"
PROCESSED_DIR = "../data/processed"
for folder in [RAW_DIR, DETAIL_DIR, PROCESSED_DIR]:
    os.makedirs(folder, exist_ok=True)

HEADERS = {
    "User-Agent": "ZHAW-AlpineSkiResearch/1.0 (academic project; repository contact)",
    "Accept-Language": "en-US,en;q=0.9",
}
REQUEST_DELAY_SECONDS = 0.8
TARGET_PER_COUNTRY = 70
MAX_LISTING_PAGES = 8
ALPINE_COUNTRY_PAGES = {
    "CH": "swiss-alps",
    "IT": "italian-alps",
    "AT": "austrian-alps",
    "FR": "french-alps",
}
COUNTRY_NAMES = {
    "CH": "Switzerland",
    "IT": "Italy",
    "AT": "Austria",
    "FR": "France",
}
# Cross-border ski domains are assigned once to a documented canonical country.
CANONICAL_COUNTRY_RULES = [
    ("portes-du-soleil", "FR"),
    ("via-lattea", "IT"),
    ("zermatt", "CH"),
    ("ischgl", "AT"),
    ("arosa-lenzerheide", "CH"),
    ("san-bernardo", "FR"),
]

session = requests.Session()
session.headers.update(HEADERS)


def normalise_text(value):
    value = str(value or "").replace("\u200b", "").replace("\u200c", "").replace("\u200d", "")
    value = value.replace("\u2060", "").replace("\ufeff", "").replace("’", "'")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", value).strip()


def to_number(value):
    if value is None:
        return np.nan
    cleaned = str(value).replace("'", "").strip()
    if re.fullmatch(r"\d{1,3}(?:,\d{3})+", cleaned):
        cleaned = cleaned.replace(",", "")
    else:
        cleaned = cleaned.replace(",", ".")
    try:
        return float(cleaned)
    except ValueError:
        return np.nan


def listing_card_text(anchor):
    """Return the smallest card ancestor containing the main resort metrics."""
    for parent in [anchor, *anchor.parents]:
        text = " ".join(parent.stripped_strings)
        has_lifts = re.search(r"\b\d+\s+ski lifts?\b", text, re.I)
        has_elevation = re.search(r"\(\s*\d[\d,.]*\s*m\s*[-–—]\s*\d[\d,.]*\s*m\s*\)", text, re.I)
        if has_lifts and has_elevation and len(text) < 1800:
            return text
    return " ".join(anchor.parent.stripped_strings)


def parse_listing_card(name, country, url, card_text):
    text = normalise_text(card_text)
    elevation = re.search(r"\(\s*([\d,.]+)\s*m\s*[-–—]\s*([\d,.]+)\s*m\s*\)", text, re.I)
    slope = re.search(r"\)\s*([\d,.]+)\s*km\b", text, re.I)
    lifts = re.search(r"\b(\d+)\s+ski lifts?\b", text, re.I)

    base_m = top_m = slope_km = lifts_count = np.nan
    if elevation:
        base_m, top_m = to_number(elevation.group(1)), to_number(elevation.group(2))
        if not (250 <= base_m < top_m <= 5000):
            base_m = top_m = np.nan
    if slope:
        slope_km = to_number(slope.group(1))
        if not (0 < slope_km <= 650):
            slope_km = np.nan
    if lifts:
        lifts_count = int(lifts.group(1))
        if not (1 <= lifts_count <= 250):
            lifts_count = np.nan

    price_raw = price_eur = np.nan
    currency = price_method = None
    chf = re.search(r"\b(?:SFr\.?|CHF)\s*([\d,.]+)", text, re.I)
    approx_eur = re.search(r"approx\.?\s*€\s*([\d,.]+)", text, re.I)
    direct_eur = re.search(r"€\s*([\d,.]+)", text)
    if chf:
        price_raw, currency = to_number(chf.group(1)), "CHF"
        if approx_eur:
            price_eur = to_number(approx_eur.group(1))
            price_method = "source_approx_eur"
        else:
            price_method = "chf_without_source_eur"
    elif direct_eur:
        price_raw = price_eur = to_number(direct_eur.group(1))
        currency, price_method = "EUR", "source_eur"

    return {
        "resort_name": name,
        "country": country,
        "url": url,
        "elevation_base_m": base_m,
        "elevation_top_m": top_m,
        "slope_km": slope_km,
        "lifts_count": lifts_count,
        "price_day_raw": price_raw,
        "currency": currency,
        "price_day_eur": price_eur,
        "rating_stars": np.nan,
        "snow_cannons": np.nan,
        "snowmaking_pct": np.nan,
        "price_method": price_method,
        "source_page_text": text,
    }


def cached_get(url, cache_path, delay=REQUEST_DELAY_SECONDS):
    """Fetch a page with a local cache and a short retry policy."""
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as handle:
            return handle.read(), "cache"
    last_error = None
    for attempt in range(3):
        try:
            response = session.get(url, timeout=35)
            response.raise_for_status()
            html = response.text
            with open(cache_path, "w", encoding="utf-8") as handle:
                handle.write(html)
            time.sleep(delay)
            return html, "downloaded"
        except requests.HTTPError as exc:
            last_error = exc
            if exc.response is not None and exc.response.status_code in {404, 410}:
                break
            time.sleep(2 ** attempt)
        except requests.RequestException as exc:
            last_error = exc
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Could not retrieve {url}: {last_error}")


def extract_villages(page_text):
    marker = "Towns/villages at the ski resort"
    start = page_text.find(marker)
    if start < 0:
        return []
    end_candidates = [
        page_text.find("Feedback for the Skiresort Team", start),
        page_text.find("Feedback/changes/error", start),
    ]
    end_candidates = [pos for pos in end_candidates if pos >= 0]
    section = page_text[start:min(end_candidates)] if end_candidates else page_text[start:start + 1200]
    matches = re.findall(
        r"([A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ0-9 .’'/-]{1,70}?)\s*\((\d+(?:\.\d+)?)\s*km\)",
        section,
    )
    villages = []
    for name, distance in matches:
        clean_name = normalise_text(name).strip(" ·•-*")
        if clean_name and clean_name not in villages:
            villages.append((clean_name, float(distance)))
    return sorted(villages, key=lambda item: item[1])


def extract_page_coordinates(soup):
    """Read coordinates only when the resort page explicitly embeds them."""
    def find_geo(value):
        if isinstance(value, dict):
            geo = value.get("geo")
            if isinstance(geo, dict) and "latitude" in geo and "longitude" in geo:
                return to_number(geo["latitude"]), to_number(geo["longitude"])
            if "latitude" in value and "longitude" in value:
                return to_number(value["latitude"]), to_number(value["longitude"])
            for child in value.values():
                found = find_geo(child)
                if found:
                    return found
        elif isinstance(value, list):
            for child in value:
                found = find_geo(child)
                if found:
                    return found
        return None

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            found = find_geo(json.loads(script.string or script.get_text()))
            if found and pd.notna(found[0]) and pd.notna(found[1]):
                lat, lon = map(float, found)
                if -90 <= lat <= 90 and -180 <= lon <= 180:
                    return lat, lon, "Skiresort JSON-LD metadata"

        except (ValueError, TypeError):
            pass

    meta = {}
    for tag in soup.find_all("meta"):
        key = tag.get("property") or tag.get("name") or tag.get("itemprop")
        value = tag.get("content")
        if key and value:
            meta[key.lower()] = value
    lat_keys = ["place:location:latitude", "latitude", "geo.position"]
    lon_keys = ["place:location:longitude", "longitude"]
    lat = lon = np.nan
    for key in lat_keys:
        if key in meta:
            if key == "geo.position" and ";" in meta[key]:
                parts = meta[key].split(";")
                lat, lon = to_number(parts[0]), to_number(parts[1])
            else:
                lat = to_number(meta[key])
            break
    for key in lon_keys:
        if key in meta:
            lon = to_number(meta[key])
            break
    if pd.notna(lat) and pd.notna(lon) and -90 <= lat <= 90 and -180 <= lon <= 180:
        return float(lat), float(lon), "Skiresort page metadata"
    return np.nan, np.nan, None


def parse_detail_page(html):
    soup = BeautifulSoup(html, "html.parser")
    text = normalise_text(soup.get_text(" ", strip=True))
    rating = re.search(r"(?:Test report|Test result|Evaluation)\s+([0-5](?:\.\d)?)\s+stars", text, re.I)
    villages = extract_villages(text)
    lat, lon, coord_method = extract_page_coordinates(soup)
    elevation = re.search(
        r"Elevation info\s+([\d,.]+)\s*m\s*[-–—]\s*([\d,.]+)\s*m", text, re.I
    )
    slopes = re.search(r"Ski slopes.*?Total:\s*([\d,.]+)\s*km", text, re.I)
    lifts = re.search(r"Ski lifts.*?Total:\s*(\d+)", text, re.I)

    price_raw = price_eur = np.nan
    currency = price_method = None
    price_section_match = re.search(r"Ski pass prices(.*?)(?:Accommodation at this ski resort|$)", text, re.I)
    price_section = price_section_match.group(1) if price_section_match else text
    adult_price = re.search(
        r"Day ticket\s+Main season\s+Adults\s*[| ]*Children\s*[| ]*(€|SFr\.?|CHF)\s*([\d,.]+)",
        price_section, re.I
    )
    if not adult_price:
        adult_price = re.search(
            r"Adults\s+Children\s+(€|SFr\.?|CHF)\s*([\d,.]+)",
            price_section, re.I
        )
    if adult_price:
        symbol, amount = adult_price.group(1), to_number(adult_price.group(2))
        if symbol == "€":
            price_raw = price_eur = amount
            currency, price_method = "EUR", "detail_source_eur"
        else:
            price_raw = amount
            currency, price_method = "CHF", "detail_source_chf_without_source_eur"

    return {
        "rating_stars": float(rating.group(1)) if rating else np.nan,
        "access_villages": "; ".join(name for name, _ in villages),
        "nearest_access_village": villages[0][0] if villages else None,
        "nearest_village_distance_km": villages[0][1] if villages else np.nan,
        "source_latitude": lat,
        "source_longitude": lon,
        "source_coordinate_method": coord_method,
        "detail_elevation_base_m": to_number(elevation.group(1)) if elevation else np.nan,
        "detail_elevation_top_m": to_number(elevation.group(2)) if elevation else np.nan,
        "detail_slope_km": to_number(slopes.group(1)) if slopes else np.nan,
        "detail_lifts_count": to_number(lifts.group(1)) if lifts else np.nan,
        "detail_price_day_raw": price_raw,
        "detail_currency": currency,
        "detail_price_day_eur": price_eur,
        "detail_price_method": price_method,
    }


def parse_snowmaking_page(html):
    text = normalise_text(BeautifulSoup(html, "html.parser").get_text(" ", strip=True))
    cannons = re.search(r"Artificial snow-making\s+([\d.,']+)\s+Snow cannons", text, re.I)
    coverage = re.search(r"([\d.,]+)\s*%\s+of the slopes have snow-making capabilities", text, re.I)
    stars = re.search(r"(?:Test result|Evaluation)\s+([0-5](?:\.\d)?)\s+stars", text, re.I)
    return {
        "snow_cannons": float(re.sub(r"[.,]", "", cannons.group(1))) if cannons else np.nan,
        "snowmaking_pct": to_number(coverage.group(1)) if coverage else np.nan,
        "snow_reliability_stars": float(stars.group(1)) if stars else np.nan,
    }


## 1. Discover resorts from the Alpine directories

Each country has a dedicated Skiresort Alpine directory. We follow its pagination, cache the HTML, and accept only canonical resort-detail URLs. Cross-border ski domains are assigned to one country using explicit slug rules so the same domain is not counted twice.

In [ ]:
candidates = []
seen_urls = set()

for country, alpine_slug in ALPINE_COUNTRY_PAGES.items():
    country_count = 0
    for page_number in range(1, MAX_LISTING_PAGES + 1):
        suffix = "" if page_number == 1 else f"page/{page_number}/"
        page_url = f"{BASE_URL}/en/ski-resorts/{alpine_slug}/{suffix}"
        cache_path = os.path.join(RAW_DIR, f"{country}_{page_number}.html")
        try:
            html, fetch_status = cached_get(page_url, cache_path)
        except RuntimeError as exc:
            print(f"[WARN] {country} listing page {page_number}: {exc}")
            break

        soup = BeautifulSoup(html, "html.parser")
        page_added = 0
        for anchor in soup.find_all("a", href=True):
            href = urljoin(BASE_URL, anchor["href"].split("?")[0])
            parsed_url = urlparse(href)
            if parsed_url.netloc not in {"www.skiresort.com", "skiresort.com"}:
                continue
            # Exclude nested test-report, photo, access and other sub-pages.
            if not re.fullmatch(r"/en/ski-resort/[^/]+/", parsed_url.path):
                continue
            canonical_url = href.rstrip("/") + "/"
            if canonical_url in seen_urls:
                continue
            name = normalise_text(anchor.get_text(" ", strip=True))
            name = re.sub(r"^ski resort\s+", "", name, flags=re.I)
            if len(name) < 3 or name.lower() in {"details", "read more", "ski resort"}:
                continue

            slug = parsed_url.path.rstrip("/").split("/")[-1]
            assigned_country = next(
                (canonical for fragment, canonical in CANONICAL_COUNTRY_RULES if fragment in slug),
                country,
            )
            card_text = listing_card_text(anchor)
            record = parse_listing_card(name, assigned_country, canonical_url, card_text)
            record["listing_country"] = country
            record["source_page"] = page_url
            record["source_retrieved_at_utc"] = pd.Timestamp.utcnow().isoformat()
            record["alpine_scope_source"] = "Skiresort.com Alpine country directory"
            record["alpine_scope_verified"] = True
            record["elevation_source"] = "Skiresort listing card" if pd.notna(record["elevation_base_m"]) else None
            record["slope_source"] = "Skiresort listing card" if pd.notna(record["slope_km"]) else None
            record["lifts_source"] = "Skiresort listing card" if pd.notna(record["lifts_count"]) else None
            record["price_source"] = "Skiresort listing card" if pd.notna(record["price_day_raw"]) else None
            record["listing_fetch_status"] = fetch_status
            candidates.append(record)
            seen_urls.add(canonical_url)
            page_added += 1

        required_fields = ["elevation_base_m", "elevation_top_m", "slope_km", "lifts_count", "price_day_raw"]
        country_count = sum(
            1 for item in candidates
            if item["country"] == country
            and sum(pd.notna(item[field]) for field in required_fields) >= 2
        )
        print(f"{country} Alpine page {page_number}: {page_added} new links; {country_count} eligible")
        if country_count >= TARGET_PER_COUNTRY:
            break
        if page_added == 0:
            break

raw = pd.DataFrame(candidates)
if raw.empty:
    raise RuntimeError("No Alpine resort records were collected. Check the Skiresort Alpine directory.")

raw.to_csv(os.path.join(PROCESSED_DIR, "resorts_scrape_candidates.csv"), index=False)
print("\nAlpine candidates:", len(raw))
print(raw.groupby("country").size().rename("candidate_count"))


## 2. Validate the listing-card parser

The parser is tested against representative EUR and CHF cards. These checks are offline and catch changes in the source's number, elevation, lift and currency formats before the live enrichment begins.

In [ ]:
sample_eur = parse_listing_card(
    "KitzSki – Kitzbühel/Kirchberg", "AT",
    "https://www.skiresort.com/en/ski-resort/kitzski/",
    "Test result 1200 m (800 m - 2000 m) 188 km 102 km 66 km 20 km 58 ski lifts € 83,-"
)
assert sample_eur["elevation_base_m"] == 800
assert sample_eur["elevation_top_m"] == 2000
assert sample_eur["slope_km"] == 188
assert sample_eur["lifts_count"] == 58
assert sample_eur["price_day_eur"] == 83

sample_chf = parse_listing_card(
    "Arosa Lenzerheide", "CH",
    "https://www.skiresort.com/en/ski-resort/arosa-lenzerheide/",
    "Test result 1636 m (1229 m - 2865 m) 225 km 110 km 87 km 28 km 43 ski lifts SFr. 89,- / approx. € 94,-"
)
assert sample_chf["price_day_raw"] == 89
assert sample_chf["currency"] == "CHF"
assert sample_chf["price_day_eur"] == 94

snow_test = parse_snowmaking_page(
    "Artificial snow-making 1200 Snow cannons 95 % of the slopes have snow-making capabilities"
)
assert snow_test["snow_cannons"] == 1200
assert snow_test["snowmaking_pct"] == 95
print("Parser smoke tests passed: EUR, CHF/EUR and snowmaking metrics.")


## 3. Enrich each selected resort from its detail pages

The main detail page provides the resort's overall Skiresort rating and nearby access villages. A separate snow-reliability page can provide the number of snow cannons and the share of slopes with snowmaking. We request both pages for the selected sample and keep a status field when a page or metric is unavailable.

In [ ]:
required_numeric = ["elevation_base_m", "elevation_top_m", "slope_km", "lifts_count", "price_day_raw"]
raw["observed_field_count"] = raw[required_numeric].notna().sum(axis=1)
raw["url_valid"] = raw["url"].str.match(r"^https://www\.skiresort\.com/en/ski-resort/[^/]+/$")
eligible = raw[
    raw["url_valid"]
    & raw["alpine_scope_verified"].eq(True)
    & (raw["observed_field_count"] >= 2)
].copy()
eligible = eligible.drop_duplicates(subset="url", keep="first")

country_counts = eligible.groupby("country").size().reindex(ALPINE_COUNTRY_PAGES, fill_value=0)
print("Eligible Alpine records by country:")
print(country_counts)
short = country_counts[country_counts < TARGET_PER_COUNTRY]
if not short.empty:
    raise RuntimeError(
        f"Fewer than {TARGET_PER_COUNTRY} source-backed Alpine resorts in: {short.to_dict()}. "
        "No non-Alpine resorts or synthetic rows will be added."
    )

selected = (
    eligible.sort_values(
        ["country", "observed_field_count", "source_page", "resort_name"],
        ascending=[True, False, True, True],
        kind="mergesort",
    )
    .groupby("country", group_keys=False)
    .head(TARGET_PER_COUNTRY)
    .copy()
    .reset_index(drop=True)
)

detail_records = []
for index, row in selected.iterrows():
    url = row["url"]
    slug = urlparse(url).path.rstrip("/").split("/")[-1]
    detail_path = os.path.join(DETAIL_DIR, f"{slug}.html")
    detail_status = "ok"
    try:
        detail_html, detail_fetch = cached_get(url, detail_path, delay=0.8)
        detail_data = parse_detail_page(detail_html)
        detail_status = detail_fetch
    except Exception as exc:
        detail_data = {
            "rating_stars": np.nan, "access_villages": None,
            "nearest_access_village": None, "nearest_village_distance_km": np.nan,
            "source_latitude": np.nan, "source_longitude": np.nan,
            "source_coordinate_method": None,
        }
        detail_status = f"error:{type(exc).__name__}"

    snow_url = url.rstrip("/") + "/test-result/snow-reliability/"
    snow_path = os.path.join(DETAIL_DIR, f"{slug}_snow_reliability.html")
    snow_status = "ok"
    try:
        snow_html, snow_fetch = cached_get(snow_url, snow_path, delay=0.8)
        snow_data = parse_snowmaking_page(snow_html)
        snow_status = snow_fetch
    except Exception as exc:
        snow_data = {
            "snow_cannons": np.nan, "snowmaking_pct": np.nan,
            "snow_reliability_stars": np.nan,
        }
        snow_status = f"unavailable:{type(exc).__name__}"

    detail_records.append({
        **detail_data,
        **snow_data,
        "detail_page_url": url,
        "snowmaking_page_url": snow_url,
        "detail_fetch_status": detail_status,
        "snowmaking_fetch_status": snow_status,
        "detail_retrieved_at_utc": pd.Timestamp.utcnow().isoformat(),
    })
    if (index + 1) % 20 == 0:
        print(f"Enriched {index + 1}/{len(selected)} resort detail pages")

details = pd.DataFrame(detail_records)
selected = selected.drop(columns=["rating_stars", "snow_cannons", "snowmaking_pct"], errors="ignore")
selected = pd.concat([selected.reset_index(drop=True), details], axis=1)

# Use resort-detail values only where the listing card did not publish a value.
for field in ["elevation_base_m", "elevation_top_m"]:
    detail_field = f"detail_{field}"
    missing = selected[field].isna() & selected[detail_field].notna()
    selected.loc[missing, field] = selected.loc[missing, detail_field]
    selected.loc[missing, "elevation_source"] = "Skiresort detail page"

for field, detail_field, source_field in [
    ("slope_km", "detail_slope_km", "slope_source"),
    ("lifts_count", "detail_lifts_count", "lifts_source"),
]:
    missing = selected[field].isna() & selected[detail_field].notna()
    selected.loc[missing, field] = selected.loc[missing, detail_field]
    selected.loc[missing, source_field] = "Skiresort detail page"

missing_price = selected["price_day_raw"].isna() & selected["detail_price_day_raw"].notna()
selected.loc[missing_price, "price_day_raw"] = selected.loc[missing_price, "detail_price_day_raw"]
selected.loc[missing_price, "currency"] = selected.loc[missing_price, "detail_currency"]
selected.loc[missing_price, "price_day_eur"] = selected.loc[missing_price, "detail_price_day_eur"]
selected.loc[missing_price, "price_method"] = selected.loc[missing_price, "detail_price_method"]
selected.loc[missing_price, "price_source"] = "Skiresort detail page"

selected["observed_field_count"] = selected[
    ["elevation_base_m", "elevation_top_m", "slope_km", "lifts_count", "price_day_raw"]
].notna().sum(axis=1)


## 4. Harmonise prices and export the Alpine sample

The final table keeps the source currency and raw ticket price. For Swiss resorts, we use an approximate EUR amount only when Skiresort explicitly displays it. We do not apply a fixed exchange rate or impute missing values. Cross-border areas are counted once under their documented canonical country.

In [ ]:
selected["price_day_eur"] = pd.to_numeric(selected["price_day_eur"], errors="coerce")
selected["price_day_raw"] = pd.to_numeric(selected["price_day_raw"], errors="coerce")
selected["price_imputed"] = False
selected["manually_added"] = False
selected["scrape_source"] = "Skiresort.com Alpine country directory + resort detail pages"

output_columns = [
    "resort_name", "country", "url", "alpine_scope_verified", "alpine_scope_source",
    "elevation_base_m", "elevation_top_m", "elevation_source",
    "slope_km", "slope_source", "lifts_count", "lifts_source",
    "price_day_raw", "currency", "price_day_eur", "price_method", "price_source",
    "rating_stars", "snow_cannons", "snowmaking_pct", "snow_reliability_stars",
    "access_villages", "nearest_access_village", "nearest_village_distance_km",
    "source_latitude", "source_longitude", "source_coordinate_method",
    "detail_page_url", "snowmaking_page_url", "detail_fetch_status",
    "snowmaking_fetch_status", "detail_retrieved_at_utc",
    "price_imputed", "manually_added", "observed_field_count",
    "scrape_source", "source_page", "source_retrieved_at_utc",
]
clean = selected[output_columns].sort_values(["country", "resort_name"]).reset_index(drop=True)
clean.to_csv(os.path.join(PROCESSED_DIR, "resorts_clean.csv"), index=False)

print("Final Alpine dataset:", len(clean))
print("Country allocation:")
print(clean.groupby("country").size())
print("\nMissing values:")
print(clean[[
    "elevation_base_m", "elevation_top_m", "slope_km", "lifts_count",
    "price_day_eur", "rating_stars", "snow_cannons", "snowmaking_pct",
    "nearest_access_village",
]].isna().sum())
display(clean.head(10))


## 5. Final quality checks

These checks verify the Alpine source scope, the balanced country sample, unique resort URLs, and the absence of manual rows or price imputation. The coverage report makes remaining gaps visible so that they can be reviewed rather than silently filled.

In [ ]:
summary = clean.groupby("country").agg(
    resorts=("resort_name", "count"),
    mean_slope_km=("slope_km", "mean"),
    median_price_eur=("price_day_eur", "median"),
    missing_prices=("price_day_eur", lambda values: int(values.isna().sum())),
    missing_ratings=("rating_stars", lambda values: int(values.isna().sum())),
    missing_snow_cannons=("snow_cannons", lambda values: int(values.isna().sum())),
)
print(summary.round(2))
print("\nDuplicate URLs:", int(clean["url"].duplicated().sum()))
print("Non-Alpine source flags:", int((~clean["alpine_scope_verified"]).sum()))
print("Manual records:", int(clean["manually_added"].sum()))
print("Price imputation flags:", int(clean["price_imputed"].sum()))
assert len(clean) == 280
assert clean.groupby("country").size().eq(TARGET_PER_COUNTRY).all()
assert clean["url"].is_unique
assert clean["alpine_scope_verified"].all()
assert not clean["manually_added"].any()
assert not clean["price_imputed"].any()
